In [1]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql.functions import col, current_timestamp, lit, when
from etl.transformations.common import create_spark, read_latest_batch, write_clickhouse, epoch_to_timestamp

MINIO_STAGING_BUCKET = os.environ.get("MINIO_STAGING_BUCKET", "staging")

In [ ]:
spark = create_spark("load_dim_crop")

In [2]:
farm_df = read_latest_batch(
            spark,
            MINIO_STAGING_BUCKET,
            "farms",
        )

infrastructure_type_df = read_latest_batch(
            spark,
            MINIO_STAGING_BUCKET,
            "farm_infrastructure_types",
        )

growing_system_type_df = read_latest_batch(
            spark,
            MINIO_STAGING_BUCKET,
            "growing_system_types",
        )

farm_df.show()
infrastructure_type_df.show()
growing_system_type_df.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/21 10:42:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/21 10:42:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+---+----------------------+----------------------+-----------+----------+-------+------+------------------+----------+----------+
| id|infrastructure_type_id|growing_system_type_id|       name|      city|size_m2|status|growing_beds_count|created_at|updated_at|
+---+----------------------+----------------------+-----------+----------+-------+------+------------------+----------+----------+
|  1|                     2|                     2|UG Farm 001|    Berlin|265.000|ACTIVE|                93|1784621971|1784621971|
|  2|                     2|                     3|UG Farm 002|    Munich|124.000|ACTIVE|                22|1784621971|1784621971|
|  3|                     2|                     3|UG Farm 003|   Hamburg|287.000|ACTIVE|                37|1784621971|1784621971|
|  4|                     2|                     2|UG Farm 004| Frankfurt|119.000|ACTIVE|                63|1784621971|1784621971|
|  5|                     2|                     1|UG Farm 005|   Cologne|135.000|A

In [3]:
dim_df = user_df.select(
        user_df.id.alias("user_id"),
        user_df.email,
        user_df.full_name,
        user_df.is_active,
        epoch_to_timestamp(user_df.created_at).alias("created_at"),
        current_timestamp().alias("_loaded_at"),
    )

NameError: name 'user_df' is not defined

In [ ]:
dim_df.count()
dim_df.show(25)
dim_df.printSchema()

print("Rows before write:", dim_df.count())

In [ ]:
write_clickhouse(
    dim_df,
    "dim_user",
)

spark.stop()